In [ ]:
import cv2
import csv

import numpy as np


# 1. ANN cu tool ( normal vs sepia trainer model )

def loadData(smaller=False):
    inputs=[]
    outputs=[]
    outputNames=['Original','Sepia']

    with open('data/etichete.csv','r') as f:
        reader = csv.reader(f)
        next(reader) # skip header

        for row in reader:
            # row[0]=filename, row[1]=label,row[2]=img_path
            label=int(row[1])
            img_path=row[2]

            # img to features
            img=cv2.imread(img_path)

            if img is not None:
                # convert bgr to rgb
                img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

                if smaller:
                    img=cv2.resize(img,(32,32))
                else:
                    img=cv2.resize(img,(128,128))
                
                img_norm=img.flatten().astype('float32')/255.0

                # add pixel matrix to inputs and label to outputs
                inputs.append(img_norm)
                outputs.append(label)

    # convert to numpy for shuffle
    inputs=np.array(inputs)
    outputs=np.array(outputs)

    # shuffle data
    noData=len(inputs)
    permutation=np.random.permutation(noData)
    inputs=inputs[permutation]
    outputs=outputs[permutation]

    return inputs, outputs, outputNames

inputs, outputs, outputNames=loadData()

In [ ]:
from sklearn.model_selection import train_test_split

# split data
trainInputs,testInputs,trainOutputs,testOutputs=train_test_split(inputs,outputs,test_size=0.20,random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# Scalăm datele: asta ajută cei 12 neuroni să "vadă" imediat culorile
trainInputs = scaler.fit_transform(trainInputs)
testInputs = scaler.transform(testInputs)

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier

# choose model and train

clf = MLPClassifier(hidden_layer_sizes=(6,7),activation='relu',solver='adam',max_iter=99,random_state=42,verbose=True)

clf.fit(trainInputs,trainOutputs)

predictions=clf.predict(testInputs)
acc=accuracy_score(testOutputs,predictions)

print('Accuracy: ',acc)

In [ ]:
# Verificati influenta (hyper)parametrilor asupra calitatii clasificatorului antrenat.

# compare different hidden layer sizes
hidden_layer_options=[(6,7),(12,),(8,8),(16,4),(5,5,5)]

for hls in hidden_layer_options:
    clf=MLPClassifier(hidden_layer_sizes=hls,activation='relu',solver='adam',max_iter=99,random_state=42,verbose=False)
    clf.fit(trainInputs,trainOutputs)
    predictions=clf.predict(testInputs)
    acc=accuracy_score(testOutputs,predictions)
    print('Hidden Layers configuration: ',hls)
    print('Accuracy: ',acc)

In [ ]:
# compare diff solvers
solvers=['lbfgs','sgd','adam']
for solver in solvers:
    clf=MLPClassifier(hidden_layer_sizes=(12,6),activation='relu',solver=solver,max_iter=99,random_state=42,verbose=False)
    clf.fit(trainInputs,trainOutputs)
    predictions=clf.predict(testInputs)
    acc=accuracy_score(testOutputs,predictions)
    print('Solver: ',solver)
    print('Accuracy: ',acc)

In [ ]:
# compare diff activations
activations=['identity','logistic','tanh','relu']

for act in activations:
    clf=MLPClassifier(hidden_layer_sizes=(12,6),activation=act,solver='adam',max_iter=99,random_state=42,verbose=False)
    clf.fit(trainInputs,trainOutputs)
    predictions=clf.predict(testInputs)
    acc=accuracy_score(testOutputs,predictions)
    print('Activation: ',act)
    print('Accuracy: ',acc)

In [ ]:
# compare diff max_iter
max_iters=[50,100,200,300]

for mi in max_iters:
    clf=MLPClassifier(hidden_layer_sizes=(6,7),activation='relu',solver='adam',max_iter=mi,random_state=42,verbose=False, n_iter_no_change=50)
    clf.fit(trainInputs,trainOutputs)
    predictions=clf.predict(testInputs)
    acc=accuracy_score(testOutputs,predictions)
    print('Epochs/Iterations: ',mi)
    print('Accuracy: ',acc)

In [ ]:
# Manual ANN implementation
from ANNie import ANN

ann=ANN(input_size=49152,hidden_size=20,output_size=1,learning_rate=0.01,epochs=100,verbose=True)

ann.fit(trainInputs,trainOutputs)
predictions=ann.predict(testInputs)
acc=accuracy_score(testOutputs,predictions)

print('Manual ANN Accuracy: ',acc)

In [ ]:
# Manual CNN implementation

from CNNews import CNN
from sklearn.metrics import accuracy_score

inputs,outputs,outputNames=loadData(smaller=True)

from sklearn.model_selection import train_test_split

trainInputs,testInputs,trainOutputs,testOutputs=train_test_split(inputs,outputs,test_size=0.20,random_state=42)

trainInputs=trainInputs[:800]
trainOutputs=trainOutputs[:800]

cnn=CNN(verbose=True,epochs=10, input_shape=(32,32,3),num_filters=16,lr=0.0001)
cnn.fit(trainInputs,trainOutputs)
predictions=cnn.predict(testInputs)
acc=accuracy_score(testOutputs,predictions)
print(f'CNN Accuracy: {acc:.6f}')